# Full separation benchmark

This notebook builds every `speech × music × difficulty` mixture, loops music when it is shorter than speech, runs every configured Demucs model, and calculates SI-SDR improvement. Run the cells from top to bottom. Long-running stages save progress after every item and can be resumed.

In [ ]:
%load_ext autoreload
%autoreload 2

import hashlib
import json
import os
import re
import sys
import time
from dataclasses import asdict
from pathlib import Path
from statistics import mean

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import librosa
import numpy as np
import torch

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = next(
    (path for path in (NOTEBOOK_DIR, *NOTEBOOK_DIR.parents) if (path / "pyproject.toml").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not find the repository root from the current directory")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.benchmark.separation import AudioMixer
from src.separation import HTDemucs
from src.utils.AudioClass import Audio

print(f"Repository: {REPO_ROOT}")

In [ ]:
# Pick the fastest available device. MPS fallback is enabled in the import cell.
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}")

In [ ]:
# Benchmark configuration. Edit DEMUCS_MODELS to add or remove model variants.
DIFFICULTIES = {
    "easy": 5.0,
    "medium": 0.0,
    "hard": -5.0,
}
DEMUCS_MODELS = [
    "htdemucs",
    "htdemucs_ft",
]
# Other Demucs variants you may add: htdemucs_6s, hdemucs_mmi, mdx, mdx_extra, mdx_q, mdx_extra_q.

MIX_SAMPLE_RATE = 44_100
MIX_CHANNELS = 2
EVAL_SAMPLE_RATE = 16_000
PEAK_CEILING_DBFS = -1.0
OVERWRITE_MIXTURES = False
OVERWRITE_MODEL_OUTPUTS = False

BENCHMARK_ROOT = REPO_ROOT / "benchmarks" / "separation"
SPEECH_ROOT = BENCHMARK_ROOT / "sources" / "speech"
MUSIC_ROOT = BENCHMARK_ROOT / "sources" / "music"
SAMPLES_ROOT = BENCHMARK_ROOT / "samples"
RESULTS_ROOT = BENCHMARK_ROOT / "results"
WORK_ROOT = REPO_ROOT / ".data" / "separation_benchmark"
AUDIO_EXTENSIONS = {".wav", ".flac", ".mp3", ".m4a", ".ogg"}

In [ ]:
# Discover all source files. Music category is the name of its parent directory.
speech_files = sorted(
    path for path in SPEECH_ROOT.rglob("*")
    if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS
)
music_files = sorted(
    path for path in MUSIC_ROOT.glob("*/*")
    if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS
)
if not speech_files:
    raise RuntimeError(f"No speech audio found under {SPEECH_ROOT}")
if not music_files:
    raise RuntimeError(f"No music audio found under {MUSIC_ROOT}")

print(f"Speech files: {len(speech_files)}")
for path in speech_files:
    print(f"  {path.relative_to(REPO_ROOT)}")
print(f"Music files: {len(music_files)}")
for path in music_files:
    print(f"  [{path.parent.name}] {path.name}")

In [ ]:
# Preview the exact workload before writing audio or downloading model weights.
mixture_count = len(speech_files) * len(music_files) * len(DIFFICULTIES)
inference_count = mixture_count * len(DEMUCS_MODELS)
speech_hours = sum((Audio.from_file(path).duration_s or 0.0) for path in speech_files) / 3600.0
rendered_program_hours = speech_hours * len(music_files) * len(DIFFICULTIES)
model_program_hours = rendered_program_hours * len(DEMUCS_MODELS)
print(f"Mixtures to render: {mixture_count}")
print(f"Demucs runs:        {inference_count}")
print(f"Rendered duration:  {rendered_program_hours:.2f} audio hours")
print(f"Model workload:     {model_program_hours:.2f} audio hours")

In [ ]:
# Build stable definitions for every speech × music × difficulty combination.
def safe_id(value):
    return re.sub(r"[^a-zA-Z0-9_-]+", "-", value).strip("-").lower()

def stable_seed(sample_id):
    return int.from_bytes(hashlib.sha256(sample_id.encode("utf-8")).digest()[:4], "big")

definitions = []
for speech_path in speech_files:
    for music_path in music_files:
        for difficulty, target_smr_db in DIFFICULTIES.items():
            sample_id = safe_id(
                f"{speech_path.stem}__{music_path.parent.name}-{music_path.stem}__{difficulty}"
            )
            definitions.append({
                "sample_id": sample_id,
                "speech_path": str(speech_path.relative_to(REPO_ROOT)),
                "music_path": str(music_path.relative_to(REPO_ROOT)),
                "music_category": music_path.parent.name,
                "difficulty": difficulty,
                "target_smr_db": target_smr_db,
                "seed": stable_seed(sample_id),
            })

print(f"Prepared {len(definitions)} definitions")
definitions[:3]

In [ ]:
# Save the benchmark plan. This file is deterministic and can be reviewed before rendering.
benchmark_path = BENCHMARK_ROOT / "benchmark.json"
benchmark_path.write_text(
    json.dumps(definitions, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print(f"Saved {benchmark_path}")

In [ ]:
# Render all mixtures and their exact speech/music references. Short music loops automatically.
SAMPLES_ROOT.mkdir(parents=True, exist_ok=True)
render_manifest_path = SAMPLES_ROOT / "render_manifest.json"
if render_manifest_path.is_file():
    render_records = json.loads(render_manifest_path.read_text(encoding="utf-8"))
else:
    render_records = []
render_by_id = {row["sample_id"]: row for row in render_records}

mixer = AudioMixer(
    sample_rate=MIX_SAMPLE_RATE,
    channels=MIX_CHANNELS,
    peak_ceiling_dbfs=PEAK_CEILING_DBFS,
)

for index, definition in enumerate(definitions, start=1):
    sample_id = definition["sample_id"]
    sample_dir = SAMPLES_ROOT / sample_id
    expected_paths = [
        sample_dir / "speech_reference.wav",
        sample_dir / "music_reference.wav",
        sample_dir / "mixture.wav",
    ]
    if not OVERWRITE_MIXTURES and all(path.is_file() for path in expected_paths):
        print(f"[{index}/{len(definitions)}] skip {sample_id}")
        if sample_id not in render_by_id:
            render_by_id[sample_id] = {
                **definition,
                "speech_reference_path": str(expected_paths[0].relative_to(REPO_ROOT)),
                "music_reference_path": str(expected_paths[1].relative_to(REPO_ROOT)),
                "mixture_path": str(expected_paths[2].relative_to(REPO_ROOT)),
                "mixing": None,
            }
        continue

    speech = Audio.from_file(REPO_ROOT / definition["speech_path"], source_id=safe_id(Path(definition["speech_path"]).stem))
    music = Audio.from_file(REPO_ROOT / definition["music_path"], source_id=safe_id(Path(definition["music_path"]).stem))
    print(f"[{index}/{len(definitions)}] render {sample_id}")
    result = mixer.mix(
        speech,
        music,
        target_smr_db=definition["target_smr_db"],
        seed=definition["seed"],
        output_dir=sample_dir,
    )
    render_by_id[sample_id] = {
        **definition,
        "speech_reference_path": str(Path(result.speech_reference.path).relative_to(REPO_ROOT)),
        "music_reference_path": str(Path(result.music_reference.path).relative_to(REPO_ROOT)),
        "mixture_path": str(Path(result.mixture.path).relative_to(REPO_ROOT)),
        "mixing": asdict(result.parameters),
    }
    render_records = [render_by_id[key] for key in sorted(render_by_id)]
    render_manifest_path.write_text(
        json.dumps(render_records, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

render_records = [render_by_id[key] for key in sorted(render_by_id)]
render_manifest_path.write_text(
    json.dumps(render_records, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print(f"Rendered/available samples: {len(render_records)}")

In [ ]:
# Confirm the rendered corpus before starting expensive model inference.
missing_mixtures = [
    definition["sample_id"]
    for definition in definitions
    if not (SAMPLES_ROOT / definition["sample_id"] / "mixture.wav").is_file()
]
if missing_mixtures:
    raise RuntimeError(f"Missing {len(missing_mixtures)} mixtures; rerun the rendering cell")
print(f"All {len(definitions)} mixtures are ready")

In [ ]:
# Run each Demucs model on every mixture. Progress is saved after every run.
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
runs_path = RESULTS_ROOT / "demucs_runs.json"
if runs_path.is_file():
    run_records = json.loads(runs_path.read_text(encoding="utf-8"))
else:
    run_records = []
run_by_key = {(row["model"], row["sample_id"]): row for row in run_records}
total_runs = len(DEMUCS_MODELS) * len(definitions)
run_index = 0

for model_name in DEMUCS_MODELS:
    for definition in definitions:
        run_index += 1
        sample_id = definition["sample_id"]
        key = (model_name, sample_id)
        mixture_path = SAMPLES_ROOT / sample_id / "mixture.wav"
        result_dir = RESULTS_ROOT / safe_id(model_name) / sample_id
        vocals_path = result_dir / "vocals.wav"
        if not OVERWRITE_MODEL_OUTPUTS and vocals_path.is_file():
            print(f"[{run_index}/{total_runs}] skip {model_name} / {sample_id}")
            if key not in run_by_key:
                run_by_key[key] = {
                    **definition,
                    "model": model_name,
                    "status": "complete",
                    "vocals_path": str(vocals_path.relative_to(REPO_ROOT)),
                    "elapsed_s": None,
                }
            continue

        result_dir.mkdir(parents=True, exist_ok=True)
        separator = HTDemucs(
            model=model_name,
            device=DEVICE,
            two_stems="vocals",
            output_dir=result_dir,
            work_dir=WORK_ROOT / safe_id(model_name) / sample_id,
            sample_rate=EVAL_SAMPLE_RATE,
            channels=1,
        )
        print(f"[{run_index}/{total_runs}] run {model_name} / {sample_id}")
        started = time.perf_counter()
        try:
            prediction = separator.separate(Audio.from_file(mixture_path, source_id=sample_id))
            prediction_path = Path(prediction.path)
            if prediction_path.resolve() != vocals_path.resolve():
                prediction_path.replace(vocals_path)
            row = {
                **definition,
                "model": model_name,
                "status": "complete",
                "vocals_path": str(vocals_path.relative_to(REPO_ROOT)),
                "elapsed_s": time.perf_counter() - started,
            }
        except Exception as exc:
            row = {
                **definition,
                "model": model_name,
                "status": "error",
                "error": f"{type(exc).__name__}: {exc}",
                "elapsed_s": time.perf_counter() - started,
            }
            print(row["error"])
        finally:
            separator.close()

        run_by_key[key] = row
        run_records = [run_by_key[item] for item in sorted(run_by_key)]
        runs_path.write_text(
            json.dumps(run_records, ensure_ascii=False, indent=2) + "\n",
            encoding="utf-8",
        )

run_records = [run_by_key[item] for item in sorted(run_by_key)]
completed = sum(row["status"] == "complete" for row in run_records)
errors = sum(row["status"] == "error" for row in run_records)
print(f"Complete: {completed}; errors: {errors}; recorded: {len(run_records)}")

In [ ]:
# Define the scale-invariant SDR metric used for scoring.
def load_mono(path, sample_rate=EVAL_SAMPLE_RATE):
    waveform, _ = librosa.load(path, sr=sample_rate, mono=True)
    return np.asarray(waveform, dtype=np.float64)

def si_sdr_db(estimate, reference):
    length = min(len(estimate), len(reference))
    estimate = estimate[:length] - np.mean(estimate[:length])
    reference = reference[:length] - np.mean(reference[:length])
    reference_energy = np.dot(reference, reference) + 1e-12
    target = (np.dot(estimate, reference) / reference_energy) * reference
    noise = estimate - target
    return float(10.0 * np.log10((np.dot(target, target) + 1e-12) / (np.dot(noise, noise) + 1e-12)))

print("Metric functions ready")

In [ ]:
# Score every successful model output against the exact speech reference.
metrics_path = RESULTS_ROOT / "metrics.json"
metric_records = []
successful_runs = [row for row in run_records if row["status"] == "complete"]
for index, row in enumerate(successful_runs, start=1):
    sample_dir = SAMPLES_ROOT / row["sample_id"]
    reference = load_mono(sample_dir / "speech_reference.wav")
    mixture = load_mono(sample_dir / "mixture.wav")
    prediction = load_mono(REPO_ROOT / row["vocals_path"])
    mixture_score = si_sdr_db(mixture, reference)
    prediction_score = si_sdr_db(prediction, reference)
    metric_records.append({
        **row,
        "mixture_si_sdr_db": mixture_score,
        "vocals_si_sdr_db": prediction_score,
        "si_sdri_db": prediction_score - mixture_score,
    })
    print(f"[{index}/{len(successful_runs)}] {row['model']} / {row['sample_id']}: SI-SDRi={prediction_score - mixture_score:.2f} dB")
    metrics_path.write_text(
        json.dumps(metric_records, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

print(f"Saved {len(metric_records)} metric rows to {metrics_path}")

In [ ]:
# Aggregate results by model and difficulty, then save a compact summary.
summary_rows = []
for model_name in DEMUCS_MODELS:
    for difficulty in DIFFICULTIES:
        rows = [
            row for row in metric_records
            if row["model"] == model_name and row["difficulty"] == difficulty
        ]
        if not rows:
            continue
        elapsed_values = [row["elapsed_s"] for row in rows if row["elapsed_s"] is not None]
        summary_rows.append({
            "model": model_name,
            "difficulty": difficulty,
            "samples": len(rows),
            "mean_vocals_si_sdr_db": mean(row["vocals_si_sdr_db"] for row in rows),
            "mean_si_sdri_db": mean(row["si_sdri_db"] for row in rows),
            "mean_elapsed_s": mean(elapsed_values) if elapsed_values else None,
        })

summary_path = RESULTS_ROOT / "summary.json"
summary_path.write_text(
    json.dumps(summary_rows, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
for row in summary_rows:
    elapsed_text = f"{row['mean_elapsed_s']:8.1f}s" if row["mean_elapsed_s"] is not None else "     n/a"
    print(
        f"{row['model']:16} {row['difficulty']:6} "
        f"n={row['samples']:3d} SI-SDR={row['mean_vocals_si_sdr_db']:7.2f} dB "
        f"SI-SDRi={row['mean_si_sdri_db']:7.2f} dB time={elapsed_text}"
    )
print(f"Saved {summary_path}")